# AAI Internship — Data Verification
Indian Domestic Aviation Route Performance & Growth Analysis (2019–2025)

In [ ]:
import os
import sqlite3
import pandas as pd
import itertools
import warnings
warnings.filterwarnings("ignore")

# Project root = parent of this Notebooks/ folder (works from any clone location)
BASE    = os.path.dirname(os.getcwd())
DB_PATH = os.path.join(BASE, "database", "aviation.db")
CLEANED = os.path.join(BASE, "data", "cleaned")

con = sqlite3.connect(DB_PATH)
print("Connected to aviation.db")

## 1. Master Dataset

In [2]:
pd.read_sql_query("""
    SELECT COUNT(*) AS total_rows, COUNT(DISTINCT route) AS unique_routes,
           MIN(year) AS from_year, MAX(year) AS to_year
    FROM citypair
""", con)

,total_rows,unique_routes,from_year,to_year
0,46473,1410,2019,2025


In [3]:
pd.read_sql_query("""
    SELECT year, COUNT(*) AS rows, COUNT(DISTINCT route) AS routes,
           ROUND(SUM(total_pax)/1e7, 2) AS pax_crore
    FROM citypair GROUP BY year ORDER BY year
""", con)

,year,rows,routes,pax_crore
0,2019,6139,685,14.37
1,2020,4569,677,6.29
2,2021,6341,791,8.27
3,2022,6964,770,12.32
4,2023,7363,822,15.20
5,2024,7414,792,16.13
6,2025,7683,870,16.69


In [4]:
# null check — all should be 0
pd.read_sql_query("""
    SELECT
        SUM(CASE WHEN route     IS NULL THEN 1 ELSE 0 END) AS null_route,
        SUM(CASE WHEN total_pax IS NULL THEN 1 ELSE 0 END) AS null_pax,
        SUM(CASE WHEN year      IS NULL THEN 1 ELSE 0 END) AS null_year,
        SUM(CASE WHEN month     IS NULL THEN 1 ELSE 0 END) AS null_month
    FROM citypair
""", con)

,null_route,null_pax,null_year,null_month
0,0,0,0,0


In [5]:
# self-route check — should be 0
pd.read_sql_query("""
    SELECT COUNT(*) AS self_routes
    FROM citypair
    WHERE SUBSTR(route,1,INSTR(route,'-')-1) = SUBSTR(route,INSTR(route,'-')+1)
""", con)

,self_routes
0,0


In [6]:
# missing months — April 2020 is expected (COVID lockdown)
existing = pd.read_sql_query("SELECT year, month FROM citypair GROUP BY year, month", con)
all_months = ["January","February","March","April","May","June",
              "July","August","September","October","November","December"]
missing = []
for year, month in itertools.product(range(2019,2026), all_months):
    if not ((existing["year"]==year) & (existing["month"]==month)).any():
        missing.append((year, month))
pd.DataFrame(missing, columns=["year","month"])

,year,month
0,2020,April


## 2. Pillar 1 — COVID Recovery

In [7]:
covid = pd.read_csv(CLEANED + r"\covid_recovery.csv")
print(f"Shape: {covid.shape}")
covid["status"].value_counts().to_frame()

Shape: (839, 6)


,count
status,
New Route,397
Fully Recovered & Grown,185
Partially Recovered,91
Recovered,76
Lost Route,47
Critically Lagging,43


In [8]:
nat = pd.read_sql_query("""
    SELECT year, ROUND(SUM(total_pax)/1e7,2) AS pax_crore
    FROM citypair WHERE year IN (2019,2024,2025)
    GROUP BY year ORDER BY year
""", con)
baseline = nat[nat["year"]==2019]["pax_crore"].values[0]
nat["recovery_pct"] = (nat["pax_crore"]/baseline*100).round(1)
nat

,year,pax_crore,recovery_pct
0,2019,14.37,100.0
1,2024,16.13,112.2
2,2025,16.69,116.1


In [ ]:
covid[covid["status"]=="Fully Recovered & Grown"].nlargest(5,"recovery_index")[
    ["route","pax_2019","pax_2025","recovery_index","status"]]

In [ ]:
covid[covid["route"]=="CHENNAI-NAGPUR"][
    ["route","pax_2019","pax_2025","recovery_index","status"]]

## 3. Pillar 2 — Underserved Airports

In [11]:
und = pd.read_csv(CLEANED + r"\underserved_airports.csv")
print(f"Shape: {und.shape}")
und["connectivity_status"].value_counts().to_frame()

Shape: (128, 7)


,count
connectivity_status,
Small Airport Town,47
Well Connected,35
Underserved,17
Adequately Connected,17
Critically Underserved,9
Hub,3


In [12]:
und[und["population_2011"] >= 300000].sort_values("pax_per_lakh").head(10)[
    ["city","population_2011","total_pax_2025","pax_per_lakh","connectivity_status"]]

,city,population_2011,total_pax_2025,pax_per_lakh,connectivity_status
0,LUDHIANA,1618879,631,39,Critically Underserved
2,AMRAVATI,549510,6952,1264,Critically Underserved
4,JAMSHEDPUR,629659,8728,1385,Critically Underserved
5,SHOLAPUR,951118,16158,1699,Critically Underserved
6,HISSAR,301249,6408,2129,Critically Underserved
8,ROURKELA,484959,15702,3238,Critically Underserved
9,KALABURAGI,532031,18908,3554,Critically Underserved
11,KURNOOL,420748,19693,4678,Critically Underserved
12,GHAZIABAD,1636068,80685,4932,Critically Underserved
15,BHAVNAGAR,593768,32618,5491,Underserved


In [13]:
print("GOA:")
display(und[und["city"]=="GOA"][["city","pax_per_lakh","connectivity_status"]])
print("DELHI:")
display(und[und["city"]=="DELHI"][["city","pax_per_lakh","connectivity_status"]])

GOA:


,city,pax_per_lakh,connectivity_status
125,GOA,1819589,Hub


DELHI:


,city,pax_per_lakh,connectivity_status
104,DELHI,327005,Well Connected


## 4. Pillar 3 — Seasonal Analysis

In [14]:
nat_monthly = pd.read_csv(CLEANED + r"\seasonal_national.csv")
airports_s  = pd.read_csv(CLEANED + r"\seasonal_airports.csv")
routes_s    = pd.read_csv(CLEANED + r"\seasonal_routes.csv")

print(f"seasonal_national : {nat_monthly.shape}")
print(f"seasonal_airports : {airports_s.shape}")
print(f"seasonal_routes   : {routes_s.shape}")

seasonal_national : (83, 3)
seasonal_airports : (123, 8)
seasonal_routes   : (447, 7)


In [15]:
month_order = ["January","February","March","April","May","June",
               "July","August","September","October","November","December"]
nat_monthly["month_num"] = nat_monthly["month"].apply(
    lambda m: month_order.index(m)+1 if m in month_order else 0)
for yr in sorted(nat_monthly["year"].unique()):
    yr_data = nat_monthly[nat_monthly["year"]==yr]
    peak = yr_data.loc[yr_data["total_pax"].idxmax()]
    print(f"  {yr}: {peak['month']} ({peak['total_pax']:,.0f} pax)")

  2019: December (12,980,408 pax)
  2020: January (12,747,591 pax)
  2021: December (11,156,079 pax)
  2022: December (12,735,454 pax)
  2023: December (13,797,338 pax)
  2024: December (14,926,352 pax)
  2025: November (15,236,646 pax)


In [16]:
print("Most seasonal airports:")
display(airports_s.sort_values("seasonality_index",ascending=False).head(5)[
    ["city","seasonality_index","peak_month","total_pax"]])

print("Most stable airports (>500k pax):")
display(airports_s[airports_s["total_pax"] > 500000].sort_values("seasonality_index").head(5)[
    ["city","seasonality_index","peak_month","total_pax"]])

Most seasonal airports:


,city,seasonality_index,peak_month,total_pax
0,ZIRO,292.1,December,534
1,LUDHIANA,213.1,September,2400
2,SHIVAMOGGA,191.9,December,116040
3,KHAJURAHO,183.2,November,108950
4,ALIGARH,177.6,August,1408


Most stable airports (>500k pax):


,city,seasonality_index,peak_month,total_pax
122,DELHI,10.2,December,109571056
121,PUNE,12.6,December,19726382
120,MUMBAI,13.7,December,76754918
119,BHUBANESWAR,13.8,December,8911381
118,CHENNAI,14.2,December,31173944


## 5. Pillar 4 — HHI Competition

In [17]:
hhi = pd.read_csv(CLEANED + r"\hhi_competition.csv")
shr = pd.read_csv(CLEANED + r"\hhi_operator_shares.csv")
print(f"hhi_competition    : {hhi.shape}")
print(f"hhi_operator_shares: {shr.shape}")
hhi["market_structure"].value_counts().to_frame()

hhi_competition    : (594, 8)
hhi_operator_shares: (895, 5)


,count
market_structure,
Monopoly,405
Highly Concentrated,188
Moderately Concentrated,1


In [18]:
hhi.sort_values("hhi").head(5)[
    ["route","num_operators","hhi","top_operator","market_structure"]]

,route,num_operators,hhi,top_operator,market_structure
593,MUMBAI-VARANASI,5,2244.5,IndiGo,Moderately Concentrated
592,MOPA-MUMBAI,4,2578.0,IndiGo,Highly Concentrated
591,DELHI-SRINAGAR,5,2639.5,IndiGo,Highly Concentrated
590,BAGDOGRA-MUMBAI,4,2692.8,Akasa Air,Highly Concentrated
589,DELHI-PUNE,5,2746.0,IndiGo,Highly Concentrated


In [19]:
hhi[hhi["market_structure"]=="Monopoly"].sort_values(
    "total_flights_season", ascending=False).head(10)[
    ["route","top_operator","avg_weekly_frequency","hhi"]]

,route,top_operator,avg_weekly_frequency,hhi
201,CHENNAI-COIMBATORE,IndiGo,111.4,10000.0
93,BENGALURU-COIMBATORE,IndiGo,84.7,10000.0
235,CHENNAI-TIRUCHIRAPALLY,IndiGo,83.5,10000.0
321,HYDERABAD-KOCHI,IndiGo,69.6,10000.0
229,COIMBATORE-HYDERABAD,IndiGo,69.6,10000.0
234,CHENNAI-TUTICORIN,IndiGo,55.7,10000.0
15,AHMEDABAD-JAIPUR,IndiGo,55.7,10000.0
354,GOA-HYDERABAD,IndiGo,55.6,10000.0
106,KOLKATA-RANCHI,IndiGo,55.6,10000.0
158,BHUBANESWAR-HYDERABAD,IndiGo,55.6,10000.0


## 6. Cross-Pillar — Underserved Cities with Monopoly Routes

In [20]:
critical_cities = und[und["connectivity_status"].isin([
    "Critically Underserved", "Underserved"
])]["city"].tolist()

mask = hhi["route"].apply(
    lambda r: any(city in r.split("-") for city in critical_cities)
)
cross = hhi[mask][["route","num_operators","hhi","market_structure",
                   "top_operator","avg_weekly_frequency"]]
cross = cross[cross["market_structure"].isin(["Monopoly","Highly Concentrated"])]
cross = cross.sort_values("avg_weekly_frequency", ascending=False)
print(f"{len(cross)} routes in underserved cities with no competition")
cross

77 routes in underserved cities with no competition


,route,num_operators,hhi,market_structure,top_operator,avg_weekly_frequency
421,JAMNAGAR-MUMBAI,2,6800.0,Highly Concentrated,Air India,34.8
282,DELHI-LUDHIANA,1,10000.0,Monopoly,Air India,27.9
296,DELHI-NASIK,1,10000.0,Monopoly,IndiGo,27.9
165,BHAVNAGAR-NAVI MUMBAI,1,10000.0,Monopoly,IndiGo,27.9
515,KOLKATA-PURNEA,2,5000.3,Highly Concentrated,IndiGo,27.6
...,...,...,...,...,...,...
17,AHMEDABAD-HISSAR,1,10000.0,Monopoly,Alliance Air,2.0
39,AMBIKAPUR-BILASPUR,1,10000.0,Monopoly,Alliance Air,2.0
191,BILASPUR-PRAYAGRAJ,1,10000.0,Monopoly,Alliance Air,2.0
68,MUMBAI-NANDED,1,10000.0,Monopoly,Ghodawat Enterprises (Fly91),2.0


In [21]:
con.close()
print("Done.")

Done.


In [ ]:
import pandas as pd

shr = pd.read_csv(os.path.join(CLEANED, "hhi_operator_shares.csv"))

result = shr.groupby("operator")["flights_in_season"].sum().reset_index()
result["share_pct"] = (result["flights_in_season"] / result["flights_in_season"].sum() * 100).round(1)
result = result.sort_values("share_pct", ascending=False).reset_index(drop=True)
result